In [1]:
!pip install catboost xgboost lightgbm -q

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
MODEL_PATH = (
"/content/drive/MyDrive/AI_MANUAL_PROJECT/models/"
)

DATASET_PATH = (
"/content/drive/MyDrive/AI_MANUAL_PROJECT/datasets/"
)

PREDICTION_PATH = (
"/content/drive/MyDrive/AI_MANUAL_PROJECT/live_prediction/"
)

os.makedirs(
    PREDICTION_PATH,
    exist_ok=True
)

In [3]:
cat_model = joblib.load(
MODEL_PATH+
"manual_catboost.pkl"
)

xgb_model = joblib.load(
MODEL_PATH+
"manual_xgboost.pkl"
)

lgb_model = joblib.load(
MODEL_PATH+
"manual_lgbm.pkl"
)

print("Models Loaded")

Models Loaded


In [4]:
df = pd.read_csv(
DATASET_PATH+
"manual_finbert_dataset.csv"
)

df["time"] = pd.to_datetime(
df["time"]
)

df = df.sort_values(
"time"
).reset_index(drop=True)

print(df.shape)

(80232, 56)


In [5]:
encoders = {}

for col in df.columns:

    if df[col].dtype=="object":

        le = LabelEncoder()

        df[col] = le.fit_transform(
            df[col].astype(str)
        )

        encoders[col] = le

print("Encoding Complete")

Encoding Complete


In [6]:
FEATURES = [

"EMA20",
"EMA50",
"RSI",
"ATR",
"NATR",

"BB_WIDTH",
"CHAIKIN_VOL",
"VQI",

"TREND_STRENGTH",
"CHANNEL_POSITION",

"SLOPE_SIGNAL",
"POWER_SCORE",

"ACTIVE_PASSIVE",

"FINANCIAL_STRENGTH",

"pair",
"timeframe",

"MARKET_STATE",

"CANDLE_PATTERN",

"FREQUENCY_TYPE",

"h1_positive",
"h1_negative",
"h1_neutral",

"h2_positive",
"h2_negative",
"h2_neutral",

"h3_positive",
"h3_negative",
"h3_neutral",

"overall_positive",
"overall_negative",
"overall_neutral",

"news_strength",

"dominant_sentiment"

]

In [7]:
latest_data = df.iloc[-1:].copy()

X_latest = latest_data[
    FEATURES
]

print(X_latest.shape)

(1, 33)


In [8]:
cat_prob = cat_model.predict_proba(
    X_latest
)[:,1][0]

xgb_prob = xgb_model.predict_proba(
    X_latest
)[:,1][0]

lgb_prob = lgb_model.predict_proba(
    X_latest
)[:,1][0]

print("CatBoost =",round(cat_prob,4))
print("XGBoost =",round(xgb_prob,4))
print("LightGBM =",round(lgb_prob,4))

CatBoost = 0.168
XGBoost = 0.1622
LightGBM = 0.1813


In [9]:
ensemble_prob = (

      0.40*cat_prob

    + 0.35*xgb_prob

    + 0.25*lgb_prob

)

print(
"Ensemble Probability =",
round(ensemble_prob,4)
)

Ensemble Probability = 0.1693


In [10]:
threshold = 0.80

if ensemble_prob >= threshold:

    signal = "BUY"

else:

    signal = "NO TRADE"

print(signal)

NO TRADE


In [11]:
confidence = ensemble_prob*100

print(
"Confidence =",
round(confidence,2),
"%"
)

Confidence = 16.93 %


In [12]:
capital = 10000

risk_per_trade = 0.02

position_size = (

    capital

    *

    risk_per_trade

)

print(
"Position Size =",
position_size
)

Position Size = 200.0


In [13]:
report = {

"Time":
latest_data["time"].values[0],

"Signal":
signal,

"Confidence_%":
round(confidence,2),

"CatBoost":
round(cat_prob,4),

"XGBoost":
round(xgb_prob,4),

"LightGBM":
round(lgb_prob,4),

"Ensemble_Probability":
round(ensemble_prob,4),

"Position_Size":
position_size

}

report = pd.DataFrame(
    [report]
)

report

,Time,Signal,Confidence_%,CatBoost,XGBoost,LightGBM,Ensemble_Probability,Position_Size
0,2026-06-21 12:00:00,NO TRADE,16.93,0.168,0.1622,0.1813,0.1693,200.0


In [14]:
report.to_csv(

PREDICTION_PATH+

"latest_prediction.csv",

index=False

)

print("Prediction Saved")

Prediction Saved


In [15]:
print()

print("========== LIVE PREDICTION ==========")

print()

print(
"Signal:",
signal
)

print(
"Confidence:",
round(confidence,2),
"%"
)

print(
"Ensemble Probability:",
round(ensemble_prob,4)
)

print(
"Position Size:",
position_size
)


========== LIVE PREDICTION ==========

Signal: NO TRADE
Confidence: 16.93 %
Ensemble Probability: 0.1693
Position Size: 200.0


In [16]:
with open(

    PREDICTION_PATH+

    "latest_signal.txt",

    "w"

) as f:

    f.write(

        f"""
Signal : {signal}

Confidence : {round(confidence,2)} %

CatBoost : {round(cat_prob,4)}

XGBoost : {round(xgb_prob,4)}

LightGBM : {round(lgb_prob,4)}

Ensemble Probability : {round(ensemble_prob,4)}

Position Size : {position_size}

"""
)

print("Signal Text File Saved")

Signal Text File Saved


# Conclusion

The live prediction system successfully loads the trained CatBoost, XGBoost and LightGBM models and evaluates the latest market observation. Individual model probabilities are combined through weighted averaging to produce a final ensemble probability. Based on the optimized threshold obtained from backtesting, the system determines whether a trade should be executed.

For the latest market sample, the ensemble probability was 0.1693, which was below the optimal threshold of 0.80. Therefore, the system correctly generated a NO TRADE signal, avoiding a low-confidence position. Risk management was incorporated by limiting the position size to 2% of available capital. Prediction reports and signal files are automatically saved for future monitoring and deployment.

Overall, the complete project consists of signal generation, FinBERT sentiment analysis, machine learning model training, strategy backtesting and live prediction, forming a complete AI-driven algorithmic trading framework.


In [17]:
all_probs = (
      0.40*cat_model.predict_proba(df[FEATURES])[:,1]
    + 0.35*xgb_model.predict_proba(df[FEATURES])[:,1]
    + 0.25*lgb_model.predict_proba(df[FEATURES])[:,1]
)

df["ensemble_probability"] = all_probs

high_confidence_signals = df[
    df["ensemble_probability"] >= 0.80
].copy()

print(high_confidence_signals.shape)

high_confidence_signals.tail(10)

(3093, 57)


,time,open,high,low,close,vol,volCcy,volCcyQuote,confirm,EMA20,...,h2_neutral,h3_positive,h3_negative,h3_neutral,overall_positive,overall_negative,overall_neutral,dominant_sentiment,news_strength,ensemble_probability
79441,2026-06-05 16:00:00,1582.98,1612.12,1547.21,1608.40,52417.710901,8.290745e+07,8.290745e+07,1,1675.634227,...,0.844135,0.03034,0.125525,0.844135,0.03034,0.125525,0.844135,1,-0.095185,0.809405
79451,2026-06-05 18:00:00,1594.20,1598.63,1545.00,1553.17,26128.283249,4.092458e+07,4.092458e+07,1,1656.941034,...,0.844135,0.03034,0.125525,0.844135,0.03034,0.125525,0.844135,1,-0.095185,0.852654
79453,2026-06-05 18:30:00,64.01,64.30,62.63,62.92,81955.734480,5.197251e+06,5.197251e+06,1,65.198583,...,0.844135,0.03034,0.125525,0.844135,0.03034,0.125525,0.844135,1,-0.095185,0.822129
79456,2026-06-05 19:00:00,62.90,63.03,61.46,62.80,110325.540541,6.866860e+06,6.866860e+06,1,64.970146,...,0.844135,0.03034,0.125525,0.844135,0.03034,0.125525,0.844135,1,-0.095185,0.810085
79457,2026-06-05 19:00:00,1553.18,1579.96,1539.28,1573.83,27296.488471,4.253436e+07,4.253436e+07,1,1649.025697,...,0.844135,0.03034,0.125525,0.844135,0.03034,0.125525,0.844135,1,-0.095185,0.800593
79458,2026-06-05 19:00:00,62.90,63.83,61.46,63.43,162147.019694,1.013930e+07,1.013930e+07,1,65.801642,...,0.844135,0.03034,0.125525,0.844135,0.03034,0.125525,0.844135,1,-0.095185,0.867875
79459,2026-06-05 20:00:00,63.43,64.81,62.68,64.67,99048.833319,6.346144e+06,6.346144e+06,1,65.693866,...,0.844135,0.03034,0.125525,0.844135,0.03034,0.125525,0.844135,1,-0.095185,0.809100
79469,2026-06-06 01:00:00,64.68,64.86,63.78,63.84,27094.173591,1.739658e+06,1.739658e+06,1,65.124821,...,0.844135,0.03034,0.125525,0.844135,0.03034,0.125525,0.844135,1,-0.095185,0.834699
79473,2026-06-06 03:00:00,63.53,63.53,62.56,62.82,46303.164220,2.917179e+06,2.917179e+06,1,64.767892,...,0.844135,0.03034,0.125525,0.844135,0.03034,0.125525,0.844135,1,-0.095185,0.800032
79488,2026-06-06 10:00:00,62.70,62.70,61.20,61.84,65044.041821,4.019191e+06,4.019191e+06,1,63.553775,...,0.844135,0.03034,0.125525,0.844135,0.03034,0.125525,0.844135,1,-0.095185,0.800857


In [18]:
demo_trade = high_confidence_signals.iloc[-1]

print()

print("===== DEMONSTRATION TRADE =====")

print()

print("Time :", demo_trade["time"])

print("Probability :", round(
    demo_trade["ensemble_probability"],
    4
))

print("Signal : BUY")

print("Pair :", demo_trade["pair"])

print("Market State :", demo_trade["MARKET_STATE"])


===== DEMONSTRATION TRADE =====

Time : 2026-06-06 10:00:00
Probability : 0.8009
Signal : BUY
Pair : 2
Market State : 2


In [19]:
high_confidence_signals.to_csv(

    PREDICTION_PATH+

    "historical_buy_signals.csv",

    index=False

)

print("Historical Buy Signals Saved")

Historical Buy Signals Saved


In [20]:
print("===== FEATURE VALUES FOR THIS TRADE =====")

display(
    latest_data[
        FEATURES
    ].T
)

===== FEATURE VALUES FOR THIS TRADE =====


,80231
EMA20,1730.798587
EMA50,1729.855677
RSI,25.140537
ATR,5.817143
NATR,0.338385
BB_WIDTH,18.952028
CHAIKIN_VOL,43.775488
VQI,0.651067
TREND_STRENGTH,0.310274
CHANNEL_POSITION,0.087895


In [21]:
display(
    demo_trade[
        FEATURES
    ]
)

,79488
EMA20,63.553775
EMA50,65.998777
RSI,36.76333
ATR,1.388571
NATR,2.245426
BB_WIDTH,4.910363
CHAIKIN_VOL,-13.532754
VQI,0.57333
TREND_STRENGTH,2.508016
CHANNEL_POSITION,0.290756
